In [1]:
"""
Dev-KOta Lite Transformer - Word-Based Nepali Text Generator
Enhanced version with word-level tokenization and improved architecture
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import re
from collections import Counter

# ==================== TOKENIZER ====================

class WordTokenizer:
    """Word-level tokenizer for Nepali text with special tokens"""
    def __init__(self, text: str, min_freq=1):
        # Tokenize by splitting on whitespace and punctuation
        self.pattern = r'\s+|([।,!?;:\-—])'
        words = self._tokenize_text(text)
        
        # Count word frequencies
        word_counts = Counter(words)
        
        # Special tokens
        self.PAD_TOKEN = '<PAD>'
        self.UNK_TOKEN = '<UNK>'
        self.BOS_TOKEN = '<BOS>'
        self.EOS_TOKEN = '<EOS>'
        
        # Build vocabulary
        special_tokens = [self.PAD_TOKEN, self.UNK_TOKEN, self.BOS_TOKEN, self.EOS_TOKEN]
        vocab_words = [word for word, count in word_counts.items() if count >= min_freq]
        
        self.vocab = special_tokens + sorted(vocab_words)
        self.vocab_size = len(self.vocab)
        
        self.stoi = {word: i for i, word in enumerate(self.vocab)}
        self.itos = {i: word for i, word in enumerate(self.vocab)}
        
        self.pad_idx = self.stoi[self.PAD_TOKEN]
        self.unk_idx = self.stoi[self.UNK_TOKEN]
        self.bos_idx = self.stoi[self.BOS_TOKEN]
        self.eos_idx = self.stoi[self.EOS_TOKEN]
    
    def _tokenize_text(self, text: str):
        """Split text into words"""
        tokens = []
        for part in re.split(self.pattern, text):
            if part and not part.isspace():
                tokens.append(part.strip())
        return tokens
    
    def encode(self, text: str, add_special_tokens=False):
        """Encode text to token indices"""
        tokens = self._tokenize_text(text)
        indices = [self.stoi.get(token, self.unk_idx) for token in tokens]
        
        if add_special_tokens:
            indices = [self.bos_idx] + indices + [self.eos_idx]
        
        return indices
    
    def decode(self, indices, skip_special_tokens=True):
        """Decode token indices to text"""
        tokens = []
        for idx in indices:
            if skip_special_tokens and idx in [self.pad_idx, self.bos_idx, self.eos_idx]:
                continue
            token = self.itos.get(idx, self.UNK_TOKEN)
            tokens.append(token)
        
        # Reconstruct text with proper spacing
        text = ' '.join(tokens)
        # Fix spacing around punctuation
        text = re.sub(r'\s+([।,!?;:])', r'\1', text)
        return text

# ==================== EMBEDDINGS ====================

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_dim, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
    
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.embedding.embedding_dim)

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding"""
    def __init__(self, emb_dim, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_seq_len, emb_dim)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, emb_dim, 2).float() * 
                            (-math.log(10000.0) / emb_dim))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        seq_len = x.size(1)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        return self.dropout(x)

# ==================== ATTENTION ====================

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, dropout=0.1):
        super().__init__()
        assert emb_dim % num_heads == 0, "emb_dim must be divisible by num_heads"
        
        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads
        
        self.qkv = nn.Linear(emb_dim, 3 * emb_dim, bias=False)
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.attn_dropout = nn.Dropout(dropout)
        
        self.register_buffer('mask', None)
    
    def forward(self, x):
        B, T, C = x.shape
        
        # Generate Q, K, V
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, num_heads, T, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Scaled dot-product attention
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Causal mask
        if self.mask is None or self.mask.size(0) < T:
            self.mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
        
        scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        # Combine heads
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        out = self.proj(out)
        out = self.dropout(out)
        
        return out

# ==================== FEED-FORWARD ====================

class FeedForward(nn.Module):
    def __init__(self, emb_dim, ff_dim=None, dropout=0.1):
        super().__init__()
        ff_dim = ff_dim or 4 * emb_dim
        
        self.net = nn.Sequential(
            nn.Linear(emb_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, emb_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)

# ==================== TRANSFORMER BLOCK ====================

class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, num_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.attn = MultiHeadAttention(emb_dim, num_heads, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ff = FeedForward(emb_dim, dropout=dropout)
    
    def forward(self, x):
        # Pre-norm architecture (more stable)
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# ==================== TRANSFORMER MODEL ====================

class DevKotaLite(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, num_heads=8, num_layers=4, 
                 max_seq_len=256, dropout=0.1, padding_idx=0):
        super().__init__()
        
        self.token_emb = TokenEmbedding(vocab_size, emb_dim, padding_idx)
        self.pos_enc = PositionalEncoding(emb_dim, max_seq_len, dropout)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(emb_dim, num_heads, dropout) 
            for _ in range(num_layers)
        ])
        
        self.ln_final = nn.LayerNorm(emb_dim)
        self.head = nn.Linear(emb_dim, vocab_size, bias=False)
        
        # Tie weights between embedding and output layer
        self.head.weight = self.token_emb.embedding.weight
        
        self.max_seq_len = max_seq_len
        self.padding_idx = padding_idx
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, x, targets=None):
        x = self.token_emb(x)
        x = self.pos_enc(x)
        
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_final(x)
        logits = self.head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), 
                targets.view(-1),
                ignore_index=self.padding_idx
            )
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, start_tokens, max_new_tokens=50, temperature=1.0, top_k=None):
        """Generate text with optional top-k sampling"""
        self.eval()
        tokens = start_tokens.clone()
        
        for _ in range(max_new_tokens):
            # Crop context if needed
            context = tokens if tokens.size(1) <= self.max_seq_len else tokens[:, -self.max_seq_len:]
            
            # Get predictions
            logits, _ = self(context)
            logits = logits[:, -1, :] / temperature
            
            # Optional top-k filtering
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            
            tokens = torch.cat([tokens, next_token], dim=1)
        
        return tokens

# ==================== DATA PROCESSING ====================

def create_dataset(data, block_size):
    """Create overlapping sequences for training"""
    sequences = []
    targets = []
    
    for i in range(0, len(data) - block_size):
        sequences.append(data[i:i+block_size])
        targets.append(data[i+1:i+block_size+1])
    
    return torch.stack(sequences), torch.stack(targets)

class TextDataLoader:
    def __init__(self, data, batch_size, block_size, device):
        self.x, self.y = create_dataset(data, block_size)
        self.batch_size = batch_size
        self.device = device
        self.num_batches = len(self.x) // batch_size
    
    def __iter__(self):
        indices = torch.randperm(len(self.x))
        for i in range(0, len(indices) - self.batch_size, self.batch_size):
            batch_indices = indices[i:i+self.batch_size]
            yield (self.x[batch_indices].to(self.device), 
                   self.y[batch_indices].to(self.device))
    
    def __len__(self):
        return self.num_batches

# ==================== TRAINER ====================

class Trainer:
    def __init__(self, model, train_data, val_data, tokenizer, 
                 batch_size=32, block_size=64, lr=3e-4, 
                 max_epochs=10, device='cpu', warmup_steps=100):
        self.model = model.to(device)
        self.tokenizer = tokenizer
        
        self.train_loader = TextDataLoader(train_data, batch_size, block_size, device)
        self.val_loader = TextDataLoader(val_data, batch_size, block_size, device)
        
        self.optimizer = torch.optim.AdamW(
            model.parameters(), 
            lr=lr, 
            betas=(0.9, 0.98),
            eps=1e-9,
            weight_decay=0.01
        )
        
        self.max_epochs = max_epochs
        self.device = device
        self.warmup_steps = warmup_steps
        self.step_count = 0
        self.base_lr = lr
    
    def _update_lr(self):
        """Learning rate warmup"""
        if self.step_count < self.warmup_steps:
            lr = self.base_lr * (self.step_count / self.warmup_steps)
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = lr
    
    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        total_loss = 0
        count = 0
        
        for x, y in self.val_loader:
            _, loss = self.model(x, y)
            total_loss += loss.item()
            count += 1
        
        self.model.train()
        return total_loss / max(count, 1)
    
    def train(self):
        self.model.train()
        best_val_loss = float('inf')
        
        for epoch in range(self.max_epochs):
            total_loss = 0
            count = 0
            
            for x, y in self.train_loader:
                self._update_lr()
                
                self.optimizer.zero_grad()
                _, loss = self.model(x, y)
                loss.backward()
                
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                
                self.optimizer.step()
                self.step_count += 1
                
                total_loss += loss.item()
                count += 1
            
            avg_train_loss = total_loss / count
            val_loss = self.evaluate()
            
            print(f"Epoch {epoch+1}/{self.max_epochs}: "
                  f"Train Loss {avg_train_loss:.4f}, Val Loss {val_loss:.4f}")
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                print(f"  → New best validation loss!")
            
            # Generate sample
            self._generate_sample()
    
    @torch.no_grad()
    def _generate_sample(self):
        """Generate a sample during training"""
        prompt = "नमस्ते"
        start_tokens = torch.tensor(
            self.tokenizer.encode(prompt), 
            dtype=torch.long
        ).unsqueeze(0).to(self.device)
        
        generated = self.model.generate(start_tokens, max_new_tokens=20, temperature=0.8, top_k=10)
        text = self.tokenizer.decode(generated[0].tolist())
        print(f"  Sample: {text}")

# ==================== MAIN ====================

if __name__ == "__main__":
    # Set random seed for reproducibility
    torch.manual_seed(42)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    # Sample Nepali text (expand this with more data)
    with open('allinone.txt','r',encoding='utf-8') as f:
        text = f.read() # Repeat to have more training data
    
    # Initialize tokenizer
    print("\nInitializing tokenizer...")
    tokenizer = WordTokenizer(text, min_freq=1)
    print(f"Vocabulary size: {tokenizer.vocab_size}")
    
    # Encode data
    data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
    n = int(0.9 * len(data))
    train_data, val_data = data[:n], data[n:]
    print(f"Train tokens: {len(train_data)}, Val tokens: {len(val_data)}")
    
    # Initialize model
    model = DevKotaLite(
        vocab_size=tokenizer.vocab_size,
        emb_dim=128,
        num_heads=8,
        num_layers=4,
        max_seq_len=64,
        dropout=0.1,
        padding_idx=tokenizer.pad_idx
    )
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")
    
    # Train
    print("\nStarting training...")
    trainer = Trainer(
        model, train_data, val_data, tokenizer,
        batch_size=8,
        block_size=32,
        lr=3e-4,
        max_epochs=50,
        device=device
    )
    trainer.train()
    
    # Final generation
    print("\n" + "="*50)
    print("Final Generation:")
    print("="*50)
    
    prompts = ["नमस्ते", "नेपाल", "काठमाडौं"]
    for prompt in prompts:
        start_tokens = torch.tensor(
            tokenizer.encode(prompt), 
            dtype=torch.long
        ).unsqueeze(0).to(device)
        
        generated = model.generate(start_tokens, max_new_tokens=30, temperature=0.8, top_k=10)
        text = tokenizer.decode(generated[0].tolist())
        print(f"\nPrompt: {prompt}")
        print(f"Generated: {text}")

Using device: cpu

Initializing tokenizer...
Vocabulary size: 13126
Train tokens: 37748, Val tokens: 4195
Model parameters: 2,471,936

Starting training...
Epoch 1/50: Train Loss 5.8214, Val Loss 9.0329
  → New best validation loss!
  Sample: <UNK> बुझयो सुन्तला आफैँले फुटेन पहाडपारि आफैँले धीरज पुछी उत्सवको अटाली नीली अटाली बुझयो मावली कोस उत्सवको ‘लक्ष्मीप्रसाद छातीमाथि आफैँले पाउन्नन्
Epoch 2/50: Train Loss 1.9410, Val Loss 13.7391
  Sample: <UNK> कर्के आकाशमाथि पृथिवीभित्र टेको टेको चिठी पृथिवीभित्र पृथिवीभित्र टुहुरो अटाली मिसिएजस्तो कर्के चिठी दुख्ता निठूरी पृथिवीभित्र कर्के अटाली चिठी अटाली
Epoch 3/50: Train Loss 0.6444, Val Loss 11.0513
  Sample: <UNK>, पानी, रानी सब रसकी! घ. उत्तर लम्क, उत्तर लम्क! स्वागत गर्छौ नेपाली! खुस्की मोती–पोल्टो
Epoch 4/50: Train Loss 0.5559, Val Loss 9.9501
  Sample: <UNK> भन्दै मुख यो भार भविष्य! तेरो उपहार निरन्तर रोई, मागूँ! प. व्योमको गुमज प्यारको गुम्मज! यी
Epoch 5/50: Train Loss 0.6039, Val Loss 9.8678
  Sample: <UNK>, मीठो बहार, अमित सुषमा, फु